# IBM Transaction Dataset Integration with Modularity-Aware Graph Autoencoders

## Overview

This notebook demonstrates how to integrate the **IBM Transactions for Anti-Money Laundering (AML)** dataset from Kaggle with the **Modularity-Aware Graph Autoencoder** code from Guillaume Salha-Galvan's repository.

### Key Resources:
- **IBM AML Dataset**: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml
- **Modularity-Aware GAE**: https://github.com/GuillaumeSalhaGalvan/modularity_aware_gae

### Challenge:
The IBM dataset contains **tabular transaction data** while the GAE code expects **graph data** (adjacency matrices and node features). This notebook provides the transformation pipeline.

### Solution Approach:
1. **Transaction Data → Graph Construction**: Convert transactions into a bipartite or entity graph
2. **Feature Engineering**: Extract meaningful node features for accounts/entities
3. **Graph Format Conversion**: Format data for GAE input requirements
4. **AML-Specific Adaptations**: Handle temporal aspects and suspicious activity patterns

## Step 1: Setup and Dependencies

First, install the required packages for both the IBM dataset processing and the GAE framework.

In [ ]:
# Install required packages
!pip install pandas numpy scipy networkx scikit-learn python-louvain
!pip install tensorflow==1.15  # GAE requires TensorFlow 1.x

# For visualization
!pip install matplotlib seaborn plotly

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import scipy.sparse as sp
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## Step 2: Load and Explore IBM Transaction Dataset

The IBM AML dataset contains transaction records with the following key columns:
- **Account information**: Account numbers, account types
- **Transaction details**: Amount, timestamp, transaction type
- **Entity information**: Receiving party, payment format
- **Risk indicators**: Is laundering flag (target variable)

In [ ]:
def load_ibm_transaction_data(file_path):
    """
    Load IBM transaction dataset.
    
    Expected format: HI-Small_Trans.csv or similar
    Download from: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml
    """
    try:
        df = pd.read_csv(file_path)
        print(f"Dataset loaded successfully: {df.shape[0]} transactions, {df.shape[1]} features")
        return df
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        print("Please download the IBM AML dataset from Kaggle and place it in the same directory.")
        # Create a sample dataset for demonstration
        return create_sample_transaction_data()

def create_sample_transaction_data():
    """
    Create a sample transaction dataset for demonstration purposes.
    This mimics the structure of the IBM AML dataset.
    """
    np.random.seed(42)
    n_transactions = 1000
    n_accounts = 200
    
    # Generate sample data
    data = {
        'Timestamp': pd.date_range('2020-01-01', periods=n_transactions, freq='H'),
        'From Bank': np.random.choice(['Bank_A', 'Bank_B', 'Bank_C'], n_transactions),
        'Account': np.random.choice(range(1, n_accounts+1), n_transactions),
        'To Bank': np.random.choice(['Bank_A', 'Bank_B', 'Bank_C'], n_transactions),
        'Account.1': np.random.choice(range(1, n_accounts+1), n_transactions),
        'Amount Received': np.random.lognormal(8, 2, n_transactions),
        'Receiving Currency': np.random.choice(['USD', 'EUR', 'GBP'], n_transactions),
        'Amount Paid': np.random.lognormal(8, 2, n_transactions),
        'Payment Currency': np.random.choice(['USD', 'EUR', 'GBP'], n_transactions),
        'Payment Format': np.random.choice(['Cash', 'Transfer', 'Check'], n_transactions),
        'Is Laundering': np.random.choice([0, 1], n_transactions, p=[0.95, 0.05])
    }
    
    df = pd.DataFrame(data)
    print("Created sample transaction dataset for demonstration.")
    print("For actual analysis, please download the real IBM AML dataset from Kaggle.")
    return df

# Load the dataset
# Replace 'HI-Small_Trans.csv' with the actual path to your downloaded dataset
df = load_ibm_transaction_data('HI-Small_Trans.csv')

# Display basic information
print("\nDataset Info:")
print(df.info())
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Explore the dataset
print("Dataset Statistics:")
print(f"Total transactions: {len(df)}")
print(f"Unique accounts: {df['Account'].nunique() + df['Account.1'].nunique()}")
print(f"Laundering cases: {df['Is Laundering'].sum()} ({df['Is Laundering'].mean()*100:.1f}%)")

# Visualize transaction amounts
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(np.log10(df['Amount Received']), bins=30, alpha=0.7)
plt.xlabel('Log10(Amount Received)')
plt.ylabel('Frequency')
plt.title('Distribution of Transaction Amounts')

plt.subplot(1, 2, 2)
laundering_counts = df['Is Laundering'].value_counts()
plt.pie(laundering_counts.values, labels=['Normal', 'Laundering'], autopct='%1.1f%%')
plt.title('Transaction Types')
plt.tight_layout()
plt.show()

## Step 3: Transaction Data → Graph Construction

The key challenge is converting transaction data into a graph format. We have several options:

### Graph Construction Strategies:
1. **Account-to-Account Graph**: Nodes are accounts, edges are transactions
2. **Bipartite Graph**: Separate nodes for sender/receiver accounts
3. **Entity-Transaction Graph**: Both accounts and transactions as nodes
4. **Temporal Graph**: Include time-based connections

We'll implement the **Account-to-Account Graph** approach as it's most suitable for AML detection.

In [ ]:
def create_transaction_graph(df, min_transactions=2):
    """
    Convert transaction data to a NetworkX graph.
    
    Parameters:
    - df: Transaction dataframe
    - min_transactions: Minimum number of transactions for an account to be included
    
    Returns:
    - G: NetworkX graph
    - account_features: Dictionary of account features
    - account_labels: Dictionary of account risk labels
    """
    
    # Create graph
    G = nx.DiGraph()  # Directed graph for transaction direction
    
    # Track account statistics for feature engineering
    account_stats = {}
    
    print("Building transaction graph...")
    
    # Process each transaction
    for idx, row in df.iterrows():
        from_account = f"Bank_{row['From Bank']}_Acc_{row['Account']}"
        to_account = f"Bank_{row['To Bank']}_Acc_{row['Account.1']}"
        amount = row['Amount Received']
        is_laundering = row['Is Laundering']
        
        # Add edge with transaction information
        if G.has_edge(from_account, to_account):
            # Update existing edge
            G[from_account][to_account]['weight'] += amount
            G[from_account][to_account]['count'] += 1
            G[from_account][to_account]['laundering_count'] += is_laundering
        else:
            # Create new edge
            G.add_edge(from_account, to_account, 
                      weight=amount, 
                      count=1, 
                      laundering_count=is_laundering)
        
        # Update account statistics
        for account in [from_account, to_account]:
            if account not in account_stats:
                account_stats[account] = {
                    'total_amount': 0,
                    'transaction_count': 0,
                    'laundering_involvement': 0,
                    'currencies': set(),
                    'payment_formats': set()
                }
            
            account_stats[account]['total_amount'] += amount
            account_stats[account]['transaction_count'] += 1
            account_stats[account]['laundering_involvement'] += is_laundering
            account_stats[account]['currencies'].add(row['Receiving Currency'])
            account_stats[account]['payment_formats'].add(row['Payment Format'])
    
    # Filter accounts with minimum transaction threshold
    accounts_to_keep = [acc for acc, stats in account_stats.items() 
                       if stats['transaction_count'] >= min_transactions]
    
    G_filtered = G.subgraph(accounts_to_keep).copy()
    
    print(f"Graph created: {G_filtered.number_of_nodes()} nodes, {G_filtered.number_of_edges()} edges")
    
    return G_filtered, account_stats

# Create the transaction graph
G, account_stats = create_transaction_graph(df)

# Display graph statistics
print(f"\nGraph Statistics:")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Graph density: {nx.density(G):.4f}")
print(f"Is connected: {nx.is_weakly_connected(G)}")
print(f"Number of weakly connected components: {nx.number_weakly_connected_components(G)}")

## Step 4: Feature Engineering for Graph Nodes

Extract meaningful features for each account (node) that can be used by the GAE model.

In [ ]:
def extract_node_features(G, account_stats):
    """
    Extract features for each node in the graph.
    
    Features include:
    - Transaction volume and frequency
    - Network centrality measures
    - Risk indicators
    - Behavioral patterns
    """
    
    nodes = list(G.nodes())
    features = []
    labels = []
    
    print("Extracting node features...")
    
    # Calculate centrality measures
    degree_centrality = nx.degree_centrality(G)
    in_degree_centrality = nx.in_degree_centrality(G)
    out_degree_centrality = nx.out_degree_centrality(G)
    betweenness_centrality = nx.betweenness_centrality(G)
    pagerank = nx.pagerank(G)
    
    for node in nodes:
        stats = account_stats.get(node, {})
        
        # Basic transaction features
        total_amount = stats.get('total_amount', 0)
        transaction_count = stats.get('transaction_count', 0)
        avg_transaction_amount = total_amount / max(transaction_count, 1)
        
        # Risk indicators
        laundering_involvement = stats.get('laundering_involvement', 0)
        laundering_ratio = laundering_involvement / max(transaction_count, 1)
        
        # Diversity measures
        currency_diversity = len(stats.get('currencies', set()))
        payment_format_diversity = len(stats.get('payment_formats', set()))
        
        # Network features
        degree_cent = degree_centrality.get(node, 0)
        in_degree_cent = in_degree_centrality.get(node, 0)
        out_degree_cent = out_degree_centrality.get(node, 0)
        betweenness_cent = betweenness_centrality.get(node, 0)
        pagerank_score = pagerank.get(node, 0)
        
        # Incoming vs outgoing transaction patterns
        in_degree = G.in_degree(node)
        out_degree = G.out_degree(node)
        degree_ratio = out_degree / max(in_degree, 1)
        
        # Aggregate edge weights
        total_incoming_weight = sum([G[pred][node]['weight'] for pred in G.predecessors(node)])
        total_outgoing_weight = sum([G[node][succ]['weight'] for succ in G.successors(node)])
        weight_balance = total_outgoing_weight - total_incoming_weight
        
        # Compile feature vector
        feature_vector = [
            np.log1p(total_amount),  # Log-transform to handle skewness
            np.log1p(avg_transaction_amount),
            transaction_count,
            laundering_ratio,
            currency_diversity,
            payment_format_diversity,
            degree_cent,
            in_degree_cent,
            out_degree_cent,
            betweenness_cent,
            pagerank_score,
            degree_ratio,
            np.log1p(abs(weight_balance)),
            in_degree,
            out_degree
        ]
        
        features.append(feature_vector)
        labels.append(1 if laundering_involvement > 0 else 0)  # Binary label
    
    features = np.array(features)
    labels = np.array(labels)
    
    print(f"Features extracted: {features.shape[0]} nodes, {features.shape[1]} features")
    print(f"Suspicious accounts: {labels.sum()} ({labels.mean()*100:.1f}%)")
    
    return features, labels, nodes

# Extract features
features, labels, node_list = extract_node_features(G, account_stats)

# Normalize features
scaler = StandardScaler()
features_normalized = scaler.fit_transform(features)

print("\nFeature statistics (after normalization):")
print(f"Feature matrix shape: {features_normalized.shape}")
print(f"Feature mean: {features_normalized.mean():.4f}")
print(f"Feature std: {features_normalized.std():.4f}")

## Step 5: Convert to GAE-Compatible Format

The modularity-aware GAE expects:
1. **Adjacency matrix**: Sparse matrix representing graph structure
2. **Feature matrix**: Node features (can be identity matrix if no features)
3. **Labels**: For community detection evaluation

In [ ]:
def convert_to_gae_format(G, features, labels, node_list):
    """
    Convert the transaction graph to GAE-compatible format.
    
    Returns:
    - adj: Sparse adjacency matrix
    - features_sparse: Sparse feature matrix
    - labels: Node labels for evaluation
    - node_mapping: Mapping from node names to indices
    """
    
    print("Converting to GAE format...")
    
    # Create node mapping
    node_mapping = {node: i for i, node in enumerate(node_list)}
    n_nodes = len(node_list)
    
    # Create adjacency matrix
    adj_matrix = np.zeros((n_nodes, n_nodes))
    
    for edge in G.edges(data=True):
        from_idx = node_mapping[edge[0]]
        to_idx = node_mapping[edge[1]]
        weight = edge[2]['weight']
        
        # For undirected version, add both directions
        adj_matrix[from_idx, to_idx] = weight
        adj_matrix[to_idx, from_idx] = weight  # Make symmetric
    
    # Convert to sparse format
    adj_sparse = sp.csr_matrix(adj_matrix)
    
    # Convert features to sparse format
    features_sparse = sp.csr_matrix(features)
    
    print(f"Adjacency matrix shape: {adj_sparse.shape}")
    print(f"Feature matrix shape: {features_sparse.shape}")
    print(f"Adjacency matrix sparsity: {1 - adj_sparse.nnz / (adj_sparse.shape[0] * adj_sparse.shape[1]):.4f}")
    
    return adj_sparse, features_sparse, labels, node_mapping

# Convert to GAE format
adj_matrix, feature_matrix, node_labels, node_mapping = convert_to_gae_format(
    G, features_normalized, labels, node_list
)

# Save the processed data
def save_gae_data(adj_matrix, feature_matrix, node_labels, node_mapping, output_dir='gae_data'):
    """
    Save the processed data in formats compatible with GAE.
    """
    import os
    import pickle
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Save adjacency matrix
    sp.save_npz(f'{output_dir}/adjacency_matrix.npz', adj_matrix)
    
    # Save feature matrix
    sp.save_npz(f'{output_dir}/feature_matrix.npz', feature_matrix)
    
    # Save labels and mapping
    np.save(f'{output_dir}/node_labels.npy', node_labels)
    
    with open(f'{output_dir}/node_mapping.pkl', 'wb') as f:
        pickle.dump(node_mapping, f)
    
    print(f"Data saved to {output_dir}/")
    
    # Create a simple edge list file (GAE format)
    edges = []
    rows, cols = adj_matrix.nonzero()
    for i, j in zip(rows, cols):
        if i < j:  # Avoid duplicates in undirected graph
            edges.append(f"{i} {j}")
    
    with open(f'{output_dir}/edgelist.txt', 'w') as f:
        f.write('\n'.join(edges))
    
    print(f"Edge list saved with {len(edges)} edges")

save_gae_data(adj_matrix, feature_matrix, node_labels, node_mapping)

## Step 6: Integration with Modularity-Aware GAE

Now we'll create the integration code to use our processed transaction data with the GAE framework.

In [ ]:
def create_gae_input_loader():
    """
    Create a custom data loader function for the IBM transaction data
    that can be integrated with the GAE input_data.py module.
    """
    
    gae_integration_code = '''
def load_ibm_transaction_data(data_dir="gae_data"):
    """
    Load IBM transaction data processed for GAE.
    
    This function should be added to the modularity_aware_gae/input_data.py file.
    """
    import scipy.sparse as sp
    import numpy as np
    import pickle
    
    # Load adjacency matrix
    adj = sp.load_npz(f"{data_dir}/adjacency_matrix.npz")
    
    # Load feature matrix
    features = sp.load_npz(f"{data_dir}/feature_matrix.npz")
    
    return adj, features

def load_ibm_labels(data_dir="gae_data"):
    """
    Load IBM transaction labels for evaluation.
    
    This function should be added to the modularity_aware_gae/input_data.py file.
    """
    import numpy as np
    
    labels = np.load(f"{data_dir}/node_labels.npy")
    return labels
'''
    
    return gae_integration_code

# Generate the integration code
integration_code = create_gae_input_loader()
print("GAE Integration Code:")
print(integration_code)

In [ ]:
# Create a complete example script for running GAE with IBM data
gae_run_script = '''
#!/usr/bin/env python3
"""
Example script to run Modularity-Aware GAE with IBM Transaction data.

Usage:
    1. Process IBM transaction data using the notebook
    2. Copy the generated gae_data/ folder to the GAE repository
    3. Add the load_ibm_transaction_data function to input_data.py
    4. Run this script
"""

import sys
import os

# Add the path to modularity_aware_gae
sys.path.append('path/to/modularity_aware_gae')

from modularity_aware_gae.train import main as train_gae
import tensorflow as tf

# Set up command line arguments for IBM transaction data
def run_gae_with_ibm_data():
    """
    Run GAE with IBM transaction data.
    """
    
    # Configure GAE parameters for transaction data
    flags = tf.app.flags
    FLAGS = flags.FLAGS
    
    # Override default parameters
    FLAGS.dataset = 'ibm_transactions'  # You'll need to add this case to input_data.py
    FLAGS.features = True  # Use the extracted features
    FLAGS.task = 'task_2'  # Joint community detection and link prediction
    FLAGS.model = 'linear_vae'  # Start with linear VAE
    FLAGS.iterations = 300
    FLAGS.learning_rate = 0.01
    FLAGS.hidden = 32
    FLAGS.dimension = 16
    FLAGS.beta = 0.5  # Modularity regularization
    FLAGS.lamb = 0.75  # Community loss weight
    FLAGS.gamma = 0.5  # Additional regularization
    FLAGS.s_reg = 2  # Sparsity regularization
    FLAGS.fastgae = False
    FLAGS.nb_run = 1
    
    print("Running Modularity-Aware GAE with IBM Transaction Data...")
    print(f"Dataset: {FLAGS.dataset}")
    print(f"Model: {FLAGS.model}")
    print(f"Features: {FLAGS.features}")
    
    # Run the training
    train_gae()

if __name__ == "__main__":
    run_gae_with_ibm_data()
'''

# Save the run script
with open('/home/runner/work/old-ML-projects/old-ML-projects/run_gae_with_ibm_data.py', 'w') as f:
    f.write(gae_run_script)

print("GAE run script created: run_gae_with_ibm_data.py")

## Step 7: Modifications Required for GAE Repository

To integrate IBM transaction data with the GAE code, you need to make these modifications:

In [ ]:
# Generate the exact modifications needed for input_data.py
input_data_modifications = '''
# Add this case to the load_data function in modularity_aware_gae/input_data.py
# Insert after line ~46 (after the 'blogs' case):

    elif dataset == 'ibm_transactions':
        # Load IBM transaction data
        adj = sp.load_npz("../gae_data/adjacency_matrix.npz")
        features = sp.load_npz("../gae_data/feature_matrix.npz")

# Add this case to the load_labels function in modularity_aware_gae/input_data.py
# Insert after line ~94 (after the 'blogs' case):

    elif dataset == 'ibm_transactions':
        labels = np.load("../gae_data/node_labels.npy")
'''

print("Required modifications for GAE input_data.py:")
print(input_data_modifications)

# Save modification instructions
with open('/home/runner/work/old-ML-projects/old-ML-projects/gae_modifications.txt', 'w') as f:
    f.write(input_data_modifications)

print("\nModification instructions saved to: gae_modifications.txt")

## Step 8: Usage Instructions and Best Practices

### Complete Integration Workflow:

1. **Download the IBM AML Dataset**:
   - Go to: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml
   - Download `HI-Small_Trans.csv` or similar files

2. **Run this Notebook**:
   - Execute all cells to process the transaction data
   - This creates the `gae_data/` folder with processed files

3. **Set up the GAE Repository**:
   ```bash
   git clone https://github.com/GuillaumeSalhaGalvan/modularity_aware_gae
   cd modularity_aware_gae
   python setup.py install
   ```

4. **Copy Processed Data**:
   ```bash
   cp -r gae_data/ modularity_aware_gae/
   ```

5. **Modify GAE Code**:
   - Add the IBM data loading cases to `modularity_aware_gae/input_data.py`
   - Use the modifications provided in `gae_modifications.txt`

6. **Run GAE Training**:
   ```bash
   cd modularity_aware_gae
   python train.py --dataset=ibm_transactions --features=True --task=task_2 --model=linear_vae --iterations=300 --learning_rate=0.01 --hidden=32 --dimension=16 --beta=0.5 --lamb=0.75 --gamma=0.5 --s_reg=2 --fastgae=False --nb_run=1
   ```

In [ ]:
# Create a comprehensive README for the integration
readme_content = '''
# IBM Transaction Dataset + Modularity-Aware GAE Integration

This repository provides a complete pipeline for integrating the IBM Transactions for Anti-Money Laundering (AML) dataset with the Modularity-Aware Graph Autoencoder framework.

## Overview

The IBM AML dataset contains financial transaction records that need to be converted into graph format for use with Graph Neural Networks. This integration enables:

- **Anti-Money Laundering Detection**: Identify suspicious transaction patterns
- **Community Detection**: Find clusters of related accounts
- **Link Prediction**: Predict future transaction relationships
- **Anomaly Detection**: Detect unusual transaction behaviors

## Key Features

- **Automatic Graph Construction**: Convert tabular transaction data to graph format
- **Feature Engineering**: Extract meaningful account-level features
- **GAE Compatibility**: Format data for direct use with GAE models
- **AML-Specific Adaptations**: Handle financial domain requirements

## Files Generated

1. `ibm_transaction_gae_integration.ipynb` - Main processing notebook
2. `run_gae_with_ibm_data.py` - Script to run GAE with processed data
3. `gae_modifications.txt` - Required modifications for GAE code
4. `gae_data/` - Processed data in GAE-compatible format

## Next Steps

1. Download the IBM AML dataset from Kaggle
2. Run the integration notebook
3. Follow the modification instructions
4. Train GAE models for AML detection

## Citation

If you use this integration in your research, please cite:

- The original GAE paper: Salha-Galvan et al. "Modularity-Aware Graph Autoencoders for Joint Community Detection and Link Prediction" (2022)
- The IBM AML dataset: IBM "Transactions for Anti Money Laundering (AML)" on Kaggle
'''

with open('/home/runner/work/old-ML-projects/old-ML-projects/IBM_GAE_Integration_README.md', 'w') as f:
    f.write(readme_content)

print("Integration README created: IBM_GAE_Integration_README.md")
print("\n=== INTEGRATION COMPLETE ===")
print("\nAll files have been generated for IBM transaction dataset integration with GAE.")
print("Check the README file for complete usage instructions.")